# Chapter 1 — Introduction
### Notebook 3 · What *is* an ontology, and what makes one bad?

*Book reference: Sections 1.3.1–1.3.3*

The book plays 'the definition game' and then sorts ontologies into good, not-so-good and bad. We make both operational — and find that the definitions genuinely disagree about real files.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
sys.path.insert(0, str(Path.cwd()))          # so ch01_toolkit imports
import ch01_toolkit as ch1
from oe_course import ontology as ont
from oe_course.data import corpus
import pandas as pd
pd.set_option("display.width", 120)

## 1. The definition game, played with a scorecard

Section 1.3.1 lists competing definitions of *ontology*. Rather than choosing a favourite, encode each as a **testable predicate** over an artefact's metrics and apply all of them to the whole corpus. Where they disagree is where the field's arguments actually live.

In [3]:
for d in ch1.DEFINITIONS:
    print(f"{d['id']:22s} {d['gloss']}")
    print(f"{'':22s} operationalised as: {d['reads_as']}\n")

gruber-1993            an explicit specification of a conceptualisation
                       operationalised as: there are named, documented terms

borst-1997             a formal specification of a shared conceptualisation
                       operationalised as: the terms carry at least some logical commitment

guarino-1998           a logical theory accounting for the intended meaning of a vocabulary
                       operationalised as: meaning is constrained beyond mere subsumption

reasoner-pragmatic     something a reasoner can derive non-trivial consequences from
                       operationalised as: there is enough logic for classification to do work



In [4]:
graphs = {a.name: ont.load_graph(a.turtle) for a in corpus.CORPUS}
scorecard = pd.DataFrame(ch1.definition_scorecard(graphs)).set_index('artefact')
scorecard

,gruber-1993,borst-1997,guarino-1998,reasoner-pragmatic
artefact,,,,
awo,True,True,True,True
colours-vocab,True,False,False,False
animals-taxonomy,True,True,False,False
food-thesaurus,True,True,False,False
broken-cycle,True,True,False,False
species-punning,True,True,False,False
staff-instance-confusion,True,True,False,False
loose-ends,True,True,False,False
opaque-ids,False,True,False,False


In [5]:
disputed = scorecard[scorecard.nunique(axis=1) > 1]
print('Artefacts the definitions DISAGREE about:\n')
print(disputed.to_string())
print('\nEach disputed row is a real argument: under Gruber it is an ontology,\n'
      'under Guarino it is not. Neither side is being careless.')

Artefacts the definitions DISAGREE about:

                          gruber-1993  borst-1997  guarino-1998  reasoner-pragmatic
artefact                                                                           
colours-vocab                    True       False         False               False
animals-taxonomy                 True        True         False               False
food-thesaurus                   True        True         False               False
broken-cycle                     True        True         False               False
species-punning                  True        True         False               False
staff-instance-confusion         True        True         False               False
loose-ends                       True        True         False               False
opaque-ids                      False        True         False               False
bare-properties                  True        True         False               False
legacy-import                    

> **Exam-style question.** `colours-vocab` counts as an ontology under Gruber-1993 and under no other definition. Is Gruber's definition too weak, or are the others too strong? Your answer should say what *work* you need the definition to do — which is the only basis on which the question can be settled.

## 2. Good, not-so-good and bad ontologies (§1.3.3)

The book's quality discussion becomes a **defect scanner**. Each detector is syntactic and explainable: it names the term that triggered it, so a finding can always be checked by hand. That is a hard requirement — a quality tool nobody can audit will be ignored the first time it is wrong.

In [6]:
for s in ont.SMELLS:
    print(f'{s.id}\n    {s.title}\n    why it matters: {s.why}\n')

subsumption-cycle
    Cycle in the class hierarchy
    why it matters: A ⊑ B ⊑ A forces the classes to be equivalent, which is almost never intended and silently collapses the taxonomy under a reasoner.

class-as-individual
    Class used as an instance
    why it matters: Conflating the class and instance levels (without deliberate punning) makes the ontology's commitments ambiguous — the classic 'is Lion a kind or a thing?' error.

individual-as-class
    Individual used in a subsumption axiom
    why it matters: rdfs:subClassOf between an individual and a class is a category error; the author meant rdf:type.

property-without-domain-or-range
    Object property lacking domain or range
    why it matters: Without domain/range the property carries no ontological commitment and a reasoner can infer nothing from its use.

no-disjointness
    Sibling classes with no disjointness
    why it matters: Absent disjointness, most modelling errors stay satisfiable and the reasoner cannot report

In [7]:
rows = []
for art in corpus.CORPUS:
    counts = ont.smell_summary(ont.load_graph(art.turtle))
    rows.append({'artefact': art.name, 'defects': sum(counts.values()), **counts})
pd.DataFrame(rows).fillna(0).set_index('artefact').astype(int)

,defects,no-disjointness,subsumption-cycle,class-as-individual,individual-as-class,undeclared-term,missing-label,property-without-domain-or-range
artefact,,,,,,,,
awo,0,0,0,0,0,0,0,0
colours-vocab,0,0,0,0,0,0,0,0
animals-taxonomy,1,1,0,0,0,0,0,0
food-thesaurus,1,1,0,0,0,0,0,0
broken-cycle,2,1,1,0,0,0,0,0
species-punning,2,1,0,1,0,0,0,0
staff-instance-confusion,1,0,0,0,1,0,0,0
loose-ends,1,0,0,0,0,1,0,0
opaque-ids,4,1,0,0,0,0,3,0


### Reading a finding

Every finding carries the evidence that produced it.

In [8]:
for f in ont.scan_smells(ont.load_graph(corpus.get('legacy-import').turtle)):
    print(f'[{f.smell}]')
    print(f'  subject: {f.subject}')
    print(f'  detail : {f.detail}\n')

[subsumption-cycle]
  subject: http://example.org/oe/Process
  detail : cycle: http://example.org/oe/Process SubClassOf http://example.org/oe/Activity SubClassOf http://example.org/oe/Process

[property-without-domain-or-range]
  subject: http://example.org/oe/involves
  detail : object property has no domain and no range; its intended use is not machine-checkable

[missing-label]
  subject: http://example.org/oe/Thing0
  detail : class has no rdfs:label; the intended meaning rests on the IRI alone



### Where the scanner is deliberately conservative

`staff-instance-confusion` has no sibling classes, so `no-disjointness` does **not** fire — there is nothing yet to be disjoint from. A detector that fired anyway would be technically defensible and practically useless, because engineers switch off tools that cry wolf. Precision is a design goal, not an accident.

In [9]:
g = ont.load_graph(corpus.get('staff-instance-confusion').turtle)
print('findings:', [f.smell for f in ont.scan_smells(g)])
print('gold labels:', sorted(corpus.get('staff-instance-confusion').gold_smells))

findings: ['individual-as-class']
gold labels: ['individual-as-class']


### Exercise 3.1 — Write a new detector

Add a detector for the **lonely child** defect: a class with exactly one direct subclass. It is a taxonomy smell — a partition into one part usually means either a missing sibling or a redundant level. Register it and run it over the corpus.

> **Hint.** Count direct `rdfs:subClassOf` children per parent; report parents with exactly one.

In [10]:
from oe_course.ontology import Finding, Smell
def detect_lonely_child(g):
    # YOUR CODE HERE: return a list of Finding objects
    return []


<details>
<summary>Solution 3.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [11]:
from collections import defaultdict
from rdflib import RDFS, URIRef
from oe_course.ontology import Finding, Smell

def detect_lonely_child(g):
    children = defaultdict(list)
    for s, _, o in g.triples((None, RDFS.subClassOf, None)):
        if isinstance(s, URIRef) and isinstance(o, URIRef):
            children[o].append(s)
    return [
        Finding('lonely-child', str(parent),
                f'exactly one direct subclass ({str(kids[0]).split(chr(35))[-1]}); '
                'a one-part partition is usually a missing sibling or a redundant level')
        for parent, kids in sorted(children.items(), key=lambda kv: str(kv[0]))
        if len(kids) == 1
    ]

lonely = Smell('lonely-child', 'Class with a single subclass',
               'A partition into one part carries no information and often marks an '
               'unfinished model.', detect_lonely_child)

hits = {a.name: len(detect_lonely_child(ont.load_graph(a.turtle))) for a in corpus.CORPUS}
print({k: v for k, v in hits.items() if v})
assert hits['animals-taxonomy'] == 1, 'Bird has exactly one subclass (Penguin)'
print('\nRegistering it would change the gold labels, so verify_corpus() would now\n'
      'fail -- correctly. Adding a detector is a breaking change to a graded rubric.')

{'awo': 2, 'animals-taxonomy': 1, 'food-thesaurus': 1, 'broken-cycle': 1, 'staff-instance-confusion': 2, 'loose-ends': 2, 'legacy-import': 2}



Registering it would change the gold labels, so verify_corpus() would now
fail -- correctly. Adding a detector is a breaking change to a graded rubric.


### Exercise 3.2 — Repair a broken ontology

Take `broken-cycle` and repair it so the scanner reports **no** defects, without deleting classes. State which real-world claim each repair encodes.

In [12]:
g = ont.load_graph(corpus.get('broken-cycle').turtle)
print('before:', ont.smell_summary(g))
# YOUR CODE HERE


before: {'subsumption-cycle': 1, 'no-disjointness': 1}


<details>
<summary>Solution 3.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [13]:
from rdflib import OWL, RDFS, URIRef
EX = 'http://example.org/oe/'
g = ont.load_graph(corpus.get('broken-cycle').turtle)
print('before:', ont.smell_summary(g))

# Repair 1: break the cycle. Vehicle is the general concept, Car the specific one,
# so the axiom 'Vehicle SubClassOf Car' is the wrong one and must go.
g.remove((URIRef(EX + 'Vehicle'), RDFS.subClassOf, URIRef(EX + 'Car')))

# Repair 2: commit to Car and Lorry being different kinds of thing.
g.add((URIRef(EX + 'Car'), OWL.disjointWith, URIRef(EX + 'Lorry')))

print('after: ', ont.smell_summary(g))
assert ont.smell_summary(g) == {}
print('level now:', ont.classify_spectrum(g)['level'])
print('\nRepair 1 asserts a direction of generality; repair 2 asserts that no\n'
      'single vehicle is both a car and a lorry. Both are claims about the world\n'
      'that a domain expert must sign off -- the tool can find the defect, but it\n'
      'cannot decide which axiom was the wrong one.')

before: {'subsumption-cycle': 1, 'no-disjointness': 1}
after:  {}
level now: formal-ontology

Repair 1 asserts a direction of generality; repair 2 asserts that no
single vehicle is both a car and a lorry. Both are claims about the world
that a domain expert must sign off -- the tool can find the defect, but it
cannot decide which axiom was the wrong one.
